# Weak-to-strong calibration pilot on Google Colab

Runs the three scripts of `w2s_calibration_pilot` (see `README.md`) on a free Colab GPU:
1. `extract_features.py`: CIFAR-10 through frozen AlexNet and DINO ViT-B/8 (the slow step);
2. `w2s_pilot.py`: weak supervisor, naive and confidence-loss students, ceiling (linear probes);
3. `make_figures.py`: reliability diagrams, summary figure, table.

**Before you start:** Runtime → Change runtime type → GPU. Have `w2s_calibration_pilot.zip` on your computer.
Colab already has torch, torchvision, numpy and matplotlib, so nothing is installed.

Not tested on Colab by the author (the original run was on a local RTX 5060). If a cell fails, send the error.

## 1. Check that a GPU is attached

In [ ]:
# Shows the GPU model and memory. If this fails, the runtime has no GPU: change the runtime type and reconnect.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Upload and unzip the folder
A file picker opens: choose `w2s_calibration_pilot.zip`.

In [ ]:
from google.colab import files     # Colab's upload/download helpers
import zipfile, os, glob

uploaded = files.upload()                                   # opens the file picker
zip_name = next(n for n in uploaded if n.endswith(".zip"))  # the zip you picked
zipfile.ZipFile(zip_name).extractall("/content")            # unpacks to /content/w2s_calibration_pilot
os.chdir(glob.glob("/content/**/w2s_pilot.py", recursive=True)[0].rsplit("/", 1)[0])  # cd into the folder
print("working directory:", os.getcwd()); print(sorted(os.listdir(".")))

## 3. Keep the original results
Step 4 overwrites `results/`. This copies the shipped results to `results_original/` for comparison.

In [ ]:
import shutil
if not os.path.exists("results_original"):
    shutil.copytree("results", "results_original")

## 4. Extract features (slow step)
Downloads CIFAR-10, AlexNet and DINO weights on first run, then writes `features/*.npy` (~0.6 GB).
About 15 min on an RTX 5060 laptop GPU; Colab time not measured. If the session disconnects, rerun this
cell: splits already saved are skipped.

In [ ]:
!python extract_features.py

## 5. Train the probes and compute every metric (about a minute on a GPU)

In [ ]:
!python w2s_pilot.py

## 6. Figures and table

In [ ]:
!python make_figures.py
import json, pandas as pd
from IPython.display import Image, display
display(pd.read_csv("results/pilot_table.csv").round(4))
for f in ["fig_pilot_summary.png", "fig_pilot_reliability.png", "fig_pilot_curves.png"]:
    display(Image(f"results/{f}"))

## 7. Compare with the original run
Small differences (fractions of a point) are expected from GPU non-determinism and float16 vs bfloat16.

In [ ]:
new = pd.read_csv("results/pilot_table.csv").set_index("model")
old = pd.read_csv("results_original/pilot_table.csv").set_index("model")
display((new - old)[["acc", "pgr", "ece"]].round(4))   # new minus original

## 8. Download your results

In [ ]:
shutil.make_archive("/content/results", "zip", "results")
files.download("/content/results.zip")